# Part 1 — Acne Detection on ACNE04 (Google Colab)

**Before running anything:** Go to **Runtime → Change runtime type → T4/L4/A100 GPU (Requires Colab Pro)**

This notebook covers all of Part 1 end-to-end in one tab:
1. Setup (clone repo, install deps, download data)
2. Data exploration
3. YOLOv8 training
4. Faster R-CNN training
5. Evaluation & comparison
6. Visualisation

Run cells top to bottom. Training cells will take ~30 min each on a T4 GPU.

---
## Section 1 — Setup

In [ ]:
# Clone repo
import os

REPO_DIR = "/content/AcneDetection"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/EvxLee/AcneDetection.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
    print("Repo already exists — pulled latest.")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies
!pip install -q roboflow ultralytics pycocotools python-dotenv PyYAML
print("Dependencies installed.")

In [ ]:
# Set Roboflow credentials — paste your API key here
import os

os.environ["ROBOFLOW_API_KEY"]   = "YOUR_API_KEY_HERE"   # ← paste your key
os.environ["ROBOFLOW_WORKSPACE"] = "evan-lee-rrndd"
os.environ["ROBOFLOW_PROJECT"]   = "acne04-detection-p8j0d"
os.environ["ROBOFLOW_VERSION"]   = "1"

with open(f"{REPO_DIR}/.env", "w") as f:
    for k in ["ROBOFLOW_API_KEY", "ROBOFLOW_WORKSPACE", "ROBOFLOW_PROJECT", "ROBOFLOW_VERSION"]:
        f.write(f"{k}={os.environ[k]}\n")
print("Credentials set.")

In [ ]:
# Download ACNE04 dataset
!python part1_detection/roboflow_loader.py --download

In [ ]:
# Verify GPU
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU detected — go to Runtime → Change runtime type → T4 GPU")

---
## Section 2 — Data Exploration

In [ ]:
import json
import random
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageDraw

%matplotlib inline

DATA_DIR = Path("data/acne04")
OUT_DIR  = Path("outputs/figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_COLORS = {
    "papules":                 "#00FFCE",
    "whitehead and blackhead": "#C7FC00",
    "pustules":                "#FE0056",
    "nodules and cysts":       "#8622FF",
}

In [ ]:
# Print dataset summary
def load_coco(split):
    with open(DATA_DIR / split / "_annotations.coco.json") as f:
        return json.load(f)

all_class_counts = Counter()
for split in ["train", "valid", "test"]:
    coco = load_coco(split)
    id_to_name = {c["id"]: c["name"] for c in coco["categories"]}
    counts = Counter(id_to_name[a["category_id"]] for a in coco["annotations"])
    all_class_counts += counts
    print(f"[{split}]  images={len(coco['images'])}  annotations={len(coco['annotations'])}")
    for cls, n in sorted(counts.items()):
        print(f"  {cls:<35} {n}")
    print()

In [ ]:
# Class distribution chart
labels = list(all_class_counts.keys())
values = [all_class_counts[l] for l in labels]
colors = [CLASS_COLORS.get(l, "#888") for l in labels]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(labels, values, color=colors, edgecolor="black", linewidth=0.6)
ax.bar_label(bars, padding=4)
ax.set_title("ACNE04 — Annotation count per class (all splits)", fontsize=13)
ax.set_ylabel("Annotations")
ax.set_xticklabels(labels, rotation=15, ha="right")
plt.tight_layout()
plt.savefig(OUT_DIR / "class_distribution.png", dpi=150)
plt.show()

In [ ]:
# Sample images with ground-truth boxes
def draw_boxes(img, annotations, id_to_name):
    img  = img.copy()
    draw = ImageDraw.Draw(img)
    for ann in annotations:
        x, y, w, h = ann["bbox"]
        label = id_to_name[ann["category_id"]]
        color = CLASS_COLORS.get(label, "#FFF")
        draw.rectangle([x, y, x+w, y+h], outline=color, width=2)
        draw.text((x+2, max(0, y-12)), label[:3].upper(), fill=color)
    return img

random.seed(42)
coco       = load_coco("train")
id_to_name = {c["id"]: c["name"] for c in coco["categories"]}
ann_map    = {}
for ann in coco["annotations"]:
    ann_map.setdefault(ann["image_id"], []).append(ann)

sample = random.sample([m for m in coco["images"] if m["id"] in ann_map], 9)

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
for i, meta in enumerate(sample):
    img = Image.open(DATA_DIR / "train" / meta["file_name"]).convert("RGB")
    img = draw_boxes(img, ann_map[meta["id"]], id_to_name)
    axes.flatten()[i].imshow(img)
    axes.flatten()[i].set_title(meta["file_name"], fontsize=7)
    axes.flatten()[i].axis("off")

patches = [mpatches.Patch(color=c, label=l) for l, c in CLASS_COLORS.items()]
fig.legend(handles=patches, loc="lower center", ncol=2, fontsize=9, bbox_to_anchor=(0.5, 0))
plt.suptitle("ACNE04 — Train samples with ground-truth annotations", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / "sample_annotations.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Section 3 — YOLOv8 Training

In [ ]:
import shutil
import yaml
from ultralytics import YOLO

YOLO_DIR  = Path("data/acne04_yolo")
YAML_PATH = YOLO_DIR / "dataset.yaml"
SPLITS    = ["train", "valid", "test"]

def coco_to_yolo(bbox, img_w, img_h):
    x, y, w, h = bbox
    return (x+w/2)/img_w, (y+h/2)/img_h, w/img_w, h/img_h

def convert_split(split, cat_to_idx):
    yolo_split = "val" if split == "valid" else split
    img_out = YOLO_DIR / "images" / yolo_split
    lbl_out = YOLO_DIR / "labels" / yolo_split
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)
    with open(DATA_DIR / split / "_annotations.coco.json") as f:
        coco = json.load(f)
    ann_map = {}
    for ann in coco["annotations"]:
        ann_map.setdefault(ann["image_id"], []).append(ann)
    for meta in coco["images"]:
        src = DATA_DIR / split / meta["file_name"]
        dst = img_out / meta["file_name"]
        if not dst.exists():
            shutil.copy2(src, dst)
        with open(lbl_out / (Path(meta["file_name"]).stem + ".txt"), "w") as f:
            for ann in ann_map.get(meta["id"], []):
                cx, cy, nw, nh = coco_to_yolo(ann["bbox"], meta["width"], meta["height"])
                f.write(f"{cat_to_idx[ann['category_id']]} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n")
    print(f"  [{split}] {len(coco['images'])} images converted")

if not YAML_PATH.exists():
    with open(DATA_DIR / "train" / "_annotations.coco.json") as f:
        coco = json.load(f)
    categories  = sorted(coco["categories"], key=lambda c: c["id"])
    class_names = [c["name"] for c in categories]
    cat_to_idx  = {c["id"]: i for i, c in enumerate(categories)}
    for split in SPLITS:
        convert_split(split, cat_to_idx)
    cfg = {"path": str(YOLO_DIR.resolve()), "train": "images/train",
           "val": "images/val", "test": "images/test",
           "nc": len(class_names), "names": class_names}
    with open(YAML_PATH, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)
    print(f"Conversion done. Classes: {class_names}")
else:
    print("Images already converted — skipping.")

# Always update path to current environment (fixes stale Mac paths in Colab)
with open(YAML_PATH) as f:
    cfg = yaml.safe_load(f)
cfg["path"] = str(YOLO_DIR.resolve())
with open(YAML_PATH, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)
print(f"YAML path set to: {cfg['path']}")


In [ ]:
# Train YOLOv8 — ~15-20 min on T4
YOLO_OUT   = Path("outputs/yolov8").resolve()   # absolute path — required for Colab
yolo_model = YOLO("yolov8s.pt")

yolo_results = yolo_model.train(
    data     = str(YAML_PATH),
    epochs   = 100,
    imgsz    = 640,
    batch    = 16,
    project  = str(YOLO_OUT),
    name     = "acne04",
    exist_ok = True,
)

In [ ]:
# YOLOv8 training curves
import pandas as pd

results_csv = YOLO_OUT / 'acne04' / 'results.csv'
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df.plot(x='epoch', y=['train/box_loss', 'val/box_loss'], ax=axes[0], title='Box Loss')
df.plot(x='epoch', y=['metrics/precision(B)', 'metrics/recall(B)'], ax=axes[1], title='Precision / Recall')
df.plot(x='epoch', y='metrics/mAP50(B)', ax=axes[2], title='mAP@50')
plt.tight_layout()
plt.savefig(OUT_DIR / 'yolov8_training_curves.png', dpi=150)
plt.show()

best = df['metrics/mAP50(B)'].idxmax()
print(f'Best epoch : {best}')
print(f'mAP@50     : {df.loc[best, "metrics/mAP50(B)"]:.4f}')
print(f'Precision  : {df.loc[best, "metrics/precision(B)"]:.4f}')
print(f'Recall     : {df.loc[best, "metrics/recall(B)"]:.4f}')


---
## Section 4 — Faster R-CNN Training

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

FRCNN_OUT = Path("outputs/faster_rcnn")
FRCNN_OUT.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

class Acne04Dataset(Dataset):
    def __init__(self, split):
        self.img_dir = DATA_DIR / split
        with open(self.img_dir / "_annotations.coco.json") as f:
            coco = json.load(f)
        cats = sorted(coco["categories"], key=lambda c: c["id"])
        self.cat_to_label = {c["id"]: i+1 for i, c in enumerate(cats)}
        self.class_names  = ["__background__"] + [c["name"] for c in cats]
        self.images  = {img["id"]: img for img in coco["images"]}
        self.ann_map = {}
        for ann in coco.get("annotations", []):
            self.ann_map.setdefault(ann["image_id"], []).append(ann)
        self.ids = list(self.images.keys())
        self.tf  = transforms.ToTensor()

    def __len__(self): return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        meta   = self.images[img_id]
        img    = Image.open(self.img_dir / meta["file_name"]).convert("RGB")
        anns   = self.ann_map.get(img_id, [])
        if anns:
            boxes  = torch.tensor([[a["bbox"][0], a["bbox"][1],
                                    a["bbox"][0]+a["bbox"][2],
                                    a["bbox"][1]+a["bbox"][3]] for a in anns], dtype=torch.float32)
            labels = torch.tensor([self.cat_to_label[a["category_id"]] for a in anns], dtype=torch.int64)
        else:
            boxes  = torch.zeros((0,4), dtype=torch.float32)
            labels = torch.zeros(0, dtype=torch.int64)
        return self.tf(img), {"boxes": boxes, "labels": labels, "image_id": torch.tensor([img_id])}

def collate_fn(batch): return tuple(zip(*batch))

train_ds = Acne04Dataset("train")
val_ds   = Acne04Dataset("valid")
print(f"Train: {len(train_ds)}  Val: {len(val_ds)}")
print(f"Classes: {train_ds.class_names}")

In [ ]:
# Train Faster R-CNN — ~15-20 min on A100
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  collate_fn=collate_fn, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, collate_fn=collate_fn, num_workers=4, pin_memory=True)

frcnn = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
frcnn.roi_heads.box_predictor = FastRCNNPredictor(
    frcnn.roi_heads.box_predictor.cls_score.in_features, 5)
frcnn.to(device)

optimizer = torch.optim.SGD(frcnn.parameters(), lr=1e-3, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

train_losses, val_losses = [], []
best_val_loss = float("inf")
EPOCHS = 20

for epoch in range(1, EPOCHS+1):
    frcnn.train()
    total = 0.0
    for imgs, targets in train_loader:
        imgs    = [i.to(device) for i in imgs]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss = sum(frcnn(imgs, targets).values())
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item()
    scheduler.step()
    avg_train = total / len(train_loader)
    train_losses.append(avg_train)

    frcnn.train()
    total = 0.0
    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs    = [i.to(device) for i in imgs]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            total  += sum(frcnn(imgs, targets).values()).item()
    avg_val = total / len(val_loader)
    val_losses.append(avg_val)

    print(f"Epoch [{epoch:02d}/{EPOCHS}]  train={avg_train:.4f}  val={avg_val:.4f}")
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(frcnn.state_dict(), FRCNN_OUT / "best.pth")
        print(f"  ✓ Checkpoint saved")

torch.save(frcnn.state_dict(), FRCNN_OUT / "last.pth")
print(f"\nDone. Best val loss: {best_val_loss:.4f}")

In [ ]:
# Faster R-CNN loss curves
plt.figure(figsize=(8,4))
plt.plot(train_losses, label="Train loss")
plt.plot(val_losses,   label="Val loss")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("Faster R-CNN — Training & Validation Loss")
plt.legend(); plt.tight_layout()
plt.savefig(OUT_DIR / "faster_rcnn_loss.png", dpi=150)
plt.show()

---
## Section 5 — Evaluation & Comparison

In [ ]:
import time, json
import numpy as np
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

CONF = 0.25  # shared confidence threshold for both models

ann_path_test = DATA_DIR / 'test' / '_annotations.coco.json'
coco_gt_test  = COCO(str(ann_path_test))
with open(ann_path_test) as f:
    coco_test_data = json.load(f)

# ── YOLOv8 — Ultralytics COCO eval on test split ─────────────────────────────
best_yolo_ckpt = YOLO_OUT / 'acne04' / 'weights' / 'best.pt'
best_yolo      = YOLO(str(best_yolo_ckpt))

# Official test-set metrics (Ultralytics COCO eval, same pipeline as training validation)
# Note: training reported val-set mAP@0.5=0.2273; this is the held-out TEST set.
yolo_metrics = best_yolo.val(data=str(YAML_PATH), split='test', conf=CONF, verbose=False)
yolo_results = {
    'mAP@50':    float(yolo_metrics.box.map50),
    'mAP@50-95': float(yolo_metrics.box.map),
    'Precision': float(yolo_metrics.box.mp),
    'Recall':    float(yolo_metrics.box.mr),
}

# Inference time (average over 20 images)
_test_imgs = [DATA_DIR / 'test' / m['file_name'] for m in coco_test_data['images'][:20]]
t0 = time.perf_counter()
for _p in _test_imgs:
    best_yolo.predict(str(_p), conf=CONF, verbose=False)
yolo_results['Inference_ms'] = round((time.perf_counter() - t0) / len(_test_imgs) * 1000, 1)

# Collect all predictions at very low conf — used for PR curve & IoU distribution
yolo_all_preds = []
for img_meta in coco_test_data['images']:
    res = best_yolo.predict(str(DATA_DIR/'test'/img_meta['file_name']),
                            conf=0.001, verbose=False)[0]
    for box, cls, score in zip(res.boxes.xyxy.tolist(),
                                res.boxes.cls.tolist(),
                                res.boxes.conf.tolist()):
        x1,y1,x2,y2 = box
        yolo_all_preds.append({'image_id': img_meta['id'],
                                'bbox': [x1, y1, x2-x1, y2-y1],
                                'score': float(score),
                                'category_id': coco_test_data['categories'][int(cls)]['id']})

print(f"YOLOv8s  test-set mAP@0.5={yolo_results['mAP@50']:.4f}  "
      f"P={yolo_results['Precision']:.4f}  R={yolo_results['Recall']:.4f}  "
      f"inf={yolo_results['Inference_ms']}ms")
print(f"(Training val-set mAP@0.5 was 0.2273 — genuine train/test split gap, same eval pipeline)")

In [ ]:
# ── Faster R-CNN — pycocotools eval ──────────────────────────────────────────
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights, FastRCNNPredictor)
from torchvision import transforms
from PIL import Image

frcnn_eval = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
frcnn_eval.roi_heads.box_predictor = FastRCNNPredictor(
    frcnn_eval.roi_heads.box_predictor.cls_score.in_features, 5)
frcnn_eval.load_state_dict(torch.load(str(FRCNN_OUT / 'best.pth'),
                                       map_location=device, weights_only=False))
frcnn_eval.to(device).eval()

cats = sorted(coco_test_data['categories'], key=lambda c: c['id'])
label_to_cat = {i+1: c['id'] for i, c in enumerate(cats)}
tf_img = transforms.ToTensor()

# Collect all predictions at very low conf (for PR curve, IoU distribution)
frcnn_all_preds = []
for img_id, img_meta in coco_gt_test.imgs.items():
    img    = Image.open(DATA_DIR / 'test' / img_meta['file_name']).convert('RGB')
    tensor = tf_img(img).unsqueeze(0).to(device)
    with torch.no_grad():
        out = frcnn_eval(tensor)[0]
    for box, lbl, score in zip(out['boxes'], out['labels'], out['scores']):
        if score.item() < 0.001: continue
        x1,y1,x2,y2 = box.tolist()
        frcnn_all_preds.append({'image_id': img_id,
                                 'category_id': label_to_cat.get(lbl.item(), lbl.item()),
                                 'bbox': [x1, y1, x2-x1, y2-y1],
                                 'score': score.item()})

# Inference time
t0 = time.perf_counter()
for img_id, img_meta in list(coco_gt_test.imgs.items())[:20]:
    img = Image.open(DATA_DIR / 'test' / img_meta['file_name']).convert('RGB')
    with torch.no_grad():
        frcnn_eval(tf_img(img).unsqueeze(0).to(device))
frcnn_ms = round((time.perf_counter() - t0) / 20 * 1000, 1)

# pycocotools mAP (same COCO eval standard as YOLOv8)
coco_dt   = coco_gt_test.loadRes(frcnn_all_preds) if frcnn_all_preds else coco_gt_test.loadRes([])
evaluator = COCOeval(coco_gt_test, coco_dt, 'bbox')
evaluator.evaluate(); evaluator.accumulate(); evaluator.summarize()

# Precision and Recall at CONF threshold — computed by matching predictions to GT
# (NOT using COCO evaluator stats, which report AR@1/AR@10/AR@100, not threshold-fixed recall)
def _box_iou(a, b):
    ax1,ay1,aw,ah = a; ax2,ay2 = ax1+aw, ay1+ah
    bx1,by1,bw,bh = b; bx2,by2 = bx1+bw, by1+bh
    ix = max(0, min(ax2,bx2) - max(ax1,bx1))
    iy = max(0, min(ay2,by2) - max(ay1,by1))
    inter = ix * iy
    union = aw*ah + bw*bh - inter
    return inter/union if union > 0 else 0.0

gt_by_img = {img_id: coco_gt_test.loadAnns(coco_gt_test.getAnnIds(imgIds=img_id))
             for img_id in coco_gt_test.imgs}
total_gt  = sum(len(v) for v in gt_by_img.values())
matched   = {img_id: [False]*len(anns) for img_id, anns in gt_by_img.items()}

tp = fp = 0
for p in sorted([x for x in frcnn_all_preds if x['score'] >= CONF], key=lambda x: -x['score']):
    img_id = p['image_id']
    best_iou, best_j = 0.0, -1
    for j, gt_ann in enumerate(gt_by_img.get(img_id, [])):
        if matched[img_id][j]: continue
        iou = _box_iou(p['bbox'], gt_ann['bbox'])
        if iou > best_iou: best_iou, best_j = iou, j
    if best_iou >= 0.5 and best_j >= 0:
        matched[img_id][best_j] = True; tp += 1
    else:
        fp += 1
fn = total_gt - tp

frcnn_results = {
    'mAP@50':       float(evaluator.stats[1]),
    'mAP@50-95':    float(evaluator.stats[0]),
    'Precision':    tp / (tp + fp) if tp + fp > 0 else 0.0,
    'Recall':       tp / (tp + fn) if tp + fn > 0 else 0.0,
    'Inference_ms': frcnn_ms,
}
print(f"Faster R-CNN  mAP@0.5={frcnn_results['mAP@50']:.4f}  "
      f"P={frcnn_results['Precision']:.4f}  R={frcnn_results['Recall']:.4f}  "
      f"inf={frcnn_results['Inference_ms']}ms")
print(f"(stats[1]=mAP@0.5  stats[6]=AR@1  stats[8]=AR@100 — Recall above is at conf={CONF})")

In [ ]:
# Comparison table
import pandas as pd

rows = []
for name, res in [('YOLOv8s', yolo_results), ('Faster R-CNN', frcnn_results)]:
    rows.append({'Model': name,
                 'mAP@50':      res['mAP@50'],
                 'mAP@50-95':   res['mAP@50-95'],
                 'Precision':   res['Precision'],
                 'Recall':      res['Recall'],
                 'Inf. (ms)':   res['Inference_ms']})

df = pd.DataFrame(rows).set_index('Model')
display(df.style.format("{:.4f}", na_rep="—").highlight_max(axis=0, color="#d4edda"))

import json as _json
with open("outputs/evaluation_results.json", "w") as f:
    _json.dump({"yolov8": yolo_results, "faster_rcnn": frcnn_results}, f, indent=2)
print("Saved → outputs/evaluation_results.json")

In [ ]:
# ── Precision-Recall curves ───────────────────────────────────────────────────
import numpy as np

def pr_curve(all_preds, coco_gt_obj, iou_thresh=0.5):
    """Compute PR curve from sorted predictions against COCO GT."""    gt_by_img = {img_id: coco_gt_obj.loadAnns(coco_gt_obj.getAnnIds(imgIds=img_id))
                 for img_id in coco_gt_obj.imgs}
    total_gt = sum(len(v) for v in gt_by_img.values())
    matched  = {img_id: [False]*len(anns) for img_id, anns in gt_by_img.items()}

    tp_list = []
    for p in sorted(all_preds, key=lambda x: -x['score']):
        img_id = p['image_id']
        best_iou, best_j = 0.0, -1
        for j, gt_ann in enumerate(gt_by_img.get(img_id, [])):
            if matched[img_id][j]: continue
            iou = _box_iou(p['bbox'], gt_ann['bbox'])
            if iou > best_iou: best_iou, best_j = iou, j
        if best_iou >= iou_thresh and best_j >= 0:
            matched[img_id][best_j] = True; tp_list.append(1)
        else:
            tp_list.append(0)

    tp_c = np.cumsum(tp_list)
    fp_c = np.cumsum([1 - x for x in tp_list])
    p_curve = tp_c / (tp_c + fp_c + 1e-9)
    r_curve = tp_c / (total_gt + 1e-9)
    return p_curve, r_curve

yolo_p_curve,  yolo_r_curve  = pr_curve(yolo_all_preds,  coco_gt_test)
frcnn_p_curve, frcnn_r_curve = pr_curve(frcnn_all_preds, coco_gt_test)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(yolo_r_curve,  yolo_p_curve,
        label=f'YOLOv8s (mAP@0.5={yolo_results["mAP@50"]:.3f})',  color='#2196F3', lw=2)
ax.plot(frcnn_r_curve, frcnn_p_curve,
        label=f'Faster R-CNN (mAP@0.5={frcnn_results["mAP@50"]:.3f})', color='#FF5722', lw=2)
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title('Precision-Recall Curves — ACNE04 Test Set (IoU=0.5)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'pr_curves.png', dpi=150)
plt.show()
print('Saved → outputs/figures/pr_curves.png')

In [ ]:
# ── IoU distribution — per-detection max IoU against GT ──────────────────────

def iou_distribution(all_preds, coco_gt_obj, conf_thresh=CONF):
    gt_by_img = {img_id: coco_gt_obj.loadAnns(coco_gt_obj.getAnnIds(imgIds=img_id))
                 for img_id in coco_gt_obj.imgs}
    ious = []
    for p in all_preds:
        if p['score'] < conf_thresh: continue
        best_iou = max(
            (_box_iou(p['bbox'], gt['bbox']) for gt in gt_by_img.get(p['image_id'], [])),
            default=0.0
        )
        ious.append(best_iou)
    return np.array(ious)

yolo_ious  = iou_distribution(yolo_all_preds,  coco_gt_test)
frcnn_ious = iou_distribution(frcnn_all_preds, coco_gt_test)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
for ax, ious, name in zip(axes,
                           [yolo_ious, frcnn_ious],
                           ['YOLOv8s', 'Faster R-CNN']):
    pct = 100 * np.mean(ious >= 0.5)
    ax.hist(ious, bins=25, range=(0,1), color='#4CAF50', edgecolor='white', alpha=0.85)
    ax.axvline(0.5, color='red', linestyle='--', lw=1.5, label='IoU=0.5 threshold')
    ax.set_xlabel('Max IoU with GT box')
    ax.set_ylabel('Detections')
    ax.set_title(f'{name}\n'
                 f'{len(ious)} dets (conf≥{CONF})  '
                 f'mean={np.mean(ious):.3f}  '
                 f'{pct:.1f}% matched')
    ax.legend(fontsize=8)

plt.suptitle('IoU Distribution of Detections — ACNE04 Test Set', fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR / 'iou_distribution.png', dpi=150)
plt.show()
print(f'YOLOv8  : mean IoU={np.mean(yolo_ious):.3f},  {100*np.mean(yolo_ious>=0.5):.1f}% above 0.5')
print(f'F-RCNN  : mean IoU={np.mean(frcnn_ious):.3f}, {100*np.mean(frcnn_ious>=0.5):.1f}% above 0.5')

In [ ]:
# ── Per-lesion-density mAP analysis ──────────────────────────────────────────
# Bin test images by annotation count, compute mAP@0.5 per bin for each model
import contextlib, io

ann_count_by_img = {}
for ann in coco_test_data['annotations']:
    ann_count_by_img[ann['image_id']] = ann_count_by_img.get(ann['image_id'], 0) + 1

def bin_label(n):
    if n <= 5:  return 'Low (1-5)'
    if n <= 15: return 'Med (6-15)'
    return 'High (16+)'

bins = {'Low (1-5)': [], 'Med (6-15)': [], 'High (16+)': []}
for img_id, n in ann_count_by_img.items():
    bins[bin_label(n)].append(img_id)

print('Lesion density bins (test set):')
for k, v in bins.items():
    print(f'  {k}: {len(v)} images')

def map50_for_ids(dt_obj, gt_obj, img_ids):
    if not img_ids: return float('nan')
    ev = COCOeval(gt_obj, dt_obj, 'bbox')
    ev.params.imgIds = img_ids
    ev.evaluate(); ev.accumulate()
    with contextlib.redirect_stdout(io.StringIO()):
        ev.summarize()  # required — populates ev.stats
    return float(ev.stats[1])   # mAP@0.5

yolo_coco_dt  = coco_gt_test.loadRes(yolo_all_preds)
frcnn_coco_dt = coco_gt_test.loadRes(frcnn_all_preds)

rows_density = []
for bin_name, img_ids in bins.items():
    rows_density.append({
        'Density': bin_name,
        'Images'  : len(img_ids),
        'YOLOv8s mAP@50'    : round(map50_for_ids(yolo_coco_dt,  coco_gt_test, img_ids), 4),
        'Faster R-CNN mAP@50': round(map50_for_ids(frcnn_coco_dt, coco_gt_test, img_ids), 4),
    })

df_density = pd.DataFrame(rows_density).set_index('Density')
print()
display(df_density.style.highlight_max(axis=0, subset=['YOLOv8s mAP@50', 'Faster R-CNN mAP@50'],
                                        color='#d4edda'))

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(bins))
w = 0.35
yolo_vals  = [r['YOLOv8s mAP@50']     for r in rows_density]
frcnn_vals = [r['Faster R-CNN mAP@50'] for r in rows_density]
ax.bar(x - w/2, yolo_vals,  w, label='YOLOv8s',      color='#2196F3', alpha=0.85)
ax.bar(x + w/2, frcnn_vals, w, label='Faster R-CNN', color='#FF5722', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(list(bins.keys()))
ax.set_ylabel('mAP@0.5')
ax.set_ylim(0, max(max(yolo_vals + frcnn_vals, default=0) * 1.4, 0.05))
ax.set_title('mAP@0.5 by Lesion Density — ACNE04 Test Set')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'density_map.png', dpi=150)
plt.show()
print('Saved → outputs/figures/density_map.png')

---
## Section 6 — Visualisation

In [ ]:
random.seed(42)
with open(DATA_DIR / 'test' / '_annotations.coco.json') as f:
    coco_test = json.load(f)

id_to_name_test = {c['id']: c['name'] for c in coco_test['categories']}
label_map = {i+1: c['name'] for i, c in enumerate(sorted(coco_test['categories'], key=lambda c: c['id']))}
ann_map_test = {}
for ann in coco_test['annotations']:
    ann_map_test.setdefault(ann['image_id'], []).append(ann)

to_tensor = transforms.ToTensor()  # local alias — safe even if cell 22 wasn't run

def draw_boxes_vis(img, boxes, labels, scores=None):
    img  = img.copy(); draw = ImageDraw.Draw(img)
    for i, box in enumerate(boxes):
        x1,y1,x2,y2 = [int(v) for v in box]
        label = labels[i] if i < len(labels) else ''
        color = CLASS_COLORS.get(label, '#FF0000')
        draw.rectangle([x1,y1,x2,y2], outline=color, width=2)
        text = label[:3].upper() + (f' {scores[i]:.2f}' if scores else '')
        draw.text((x1+2, max(0,y1-12)), text, fill=color)
    return img

sample = random.sample([m for m in coco_test['images'] if m['id'] in ann_map_test], 6)
fig, axes = plt.subplots(6, 3, figsize=(15, 30))
for col, title in enumerate(['Ground Truth', 'YOLOv8s', 'Faster R-CNN']):
    axes[0][col].set_title(title, fontsize=13, fontweight='bold')

for row, meta in enumerate(sample):
    img_path = DATA_DIR / 'test' / meta['file_name']
    img      = Image.open(img_path).convert('RGB')

    # GT
    anns      = ann_map_test[meta['id']]
    gt_boxes  = [[a['bbox'][0],a['bbox'][1],a['bbox'][0]+a['bbox'][2],a['bbox'][1]+a['bbox'][3]] for a in anns]
    gt_labels = [id_to_name_test[a['category_id']] for a in anns]

    # YOLOv8
    res      = best_yolo.predict(str(img_path), conf=CONF, verbose=False)[0]
    y_boxes  = res.boxes.xyxy.tolist()
    y_labels = [res.names[int(c)] for c in res.boxes.cls.tolist()]
    y_scores = res.boxes.conf.tolist()

    # Faster R-CNN
    with torch.no_grad():
        out = frcnn_eval(to_tensor(img).unsqueeze(0).to(device))[0]
    f_boxes  = [b.tolist() for b,s in zip(out['boxes'],out['scores']) if s>=CONF]
    f_labels = [label_map.get(l.item(),str(l.item())) for l,s in zip(out['labels'],out['scores']) if s>=CONF]
    f_scores = [s.item() for s in out['scores'] if s>=CONF]

    for col, (vis_img, boxes, labels, scores) in enumerate([
        (img, gt_boxes, gt_labels, None),
        (img, y_boxes,  y_labels,  y_scores),
        (img, f_boxes,  f_labels,  f_scores),
    ]):
        axes[row][col].imshow(draw_boxes_vis(vis_img, boxes, labels, scores))
        axes[row][col].axis('off')

patches = [mpatches.Patch(color=c, label=l) for l, c in CLASS_COLORS.items()]
fig.legend(handles=patches, loc='lower center', ncol=2, fontsize=9, bbox_to_anchor=(0.5, 0))
plt.suptitle('ACNE04 Test Set — YOLOv8 vs Faster R-CNN', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / 'detection_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → outputs/figures/detection_comparison.png')
